In [7]:
import pandas as pd
import deltalake
import pyarrow as pa
import polars as pl
from azure.storage.blob import BlobServiceClient
from azure.core.credentials import AzureNamedKeyCredential

In [8]:
def download_delta_table(account_name, account_key):
    """
    Downloads Delta tables from Azure Blob Storage and saves it locally as a Parquet file.
    
    :param account_name: Azure Storage account name
    :param account_key: Azure Storage account key
    :description: This function connects to an Azure Blob Storage account, lists all containers,
                 and checks if each container contains a Delta table.

    """

    # Create Azure credentials
    account_credential = AzureNamedKeyCredential(account_name, account_key)

    # Create the BlobServiceClient object
    blob_service_client = BlobServiceClient(f"https://{account_name}.blob.core.windows.net", credential=account_credential)

    az_storage_options = {
        "AZURE_STORAGE_ACCOUNT_NAME": account_name,
        "AZURE_STORAGE_ACCOUNT_KEY": account_key,
    }

    # List all containers in the storage account
    list_of_containers = blob_service_client.list_containers(include_metadata=True)

    # Iterate through each container and check if it is a Delta table
    for account_container in list_of_containers:
        container_name = account_container['name']
        az_table_path = f"abfs://{container_name}/delta_table"

        # If it is a delta-table, add to the delta table list
        if deltalake.DeltaTable.is_deltatable(az_table_path, storage_options=az_storage_options):
            dt = deltalake.DeltaTable(az_table_path, storage_options=az_storage_options)
            history_df = pd.DataFrame(dt.history())
            history_df.to_parquet(f"dt_history/{container_name}/dt_hist_{account_name}_df.parquet")
            df = dt.to_pandas()
            df['received_at']= pd.to_datetime(df['received_at'], utc=True)
            df = df.set_index('received_at', drop=False)
            df.to_parquet(f"{container_name}/{account_name}_df.parquet")
            print(f'{container_name} is a delta table! DataFrame saved as {container_name}/{account_name}_df.parquet')
        else:
            # If it is not a delta table, print the message
            print(f'{container_name} is not a delta table!')


In [ ]:
# Downloads the reference Delta lake
experiments_data_lake_info = [
    {
        "account_name": "?????????????",
        "account_key": "??????????????",
    },
    {
        "account_name": "??????????????",
        "account_key": "??????????????????????",
    },
    {
        "account_name": "??????????????",
        "account_key": "??????????????????????",
    },
    {
        "account_name": "??????????????",
        "account_key": "??????????????????????",
    },
    {
        "account_name": "??????????????",
        "account_key": "??????????????????????",
    },
    {
        "account_name": "??????????????",
        "account_key": "??????????????????????",
    },
    {
        "account_name": "??????????????",
        "account_key": "??????????????????????",
    },
]

# Download Delta tables from each account
for info in experiments_data_lake_info:
    try:
        account_name = info['account_name']
        account_key = info['account_key']
        download_delta_table(account_name, account_key)
    except Exception as e:
        print(f"Error downloading from account {account_name}: {e}")

print("Download complete!")